## **OPTIMIZACIÓN DE PARÁMETROS EGNA**

#### **NLSE**

In [1]:
import numpy as np
import sys
import os
# Get the absolute path of the main directory
main_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# Add the main directory to sys.path
sys.path.append(main_dir)
from OptFunctions.NonLinearSE import *
import time
#Para Bayesian optimization de parámetros
from skopt import gp_minimize
from skopt.space import Integer, Real, Categorical
from skopt.utils import use_named_args
#Para usar EGNA
from EDAspy.optimization import EGNA
from EDAspy.optimization import UMDAc
from EDAspy.optimization import EMNA
#Para procesos Gaussianos
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
import pickle

n_variables = [50, 425, 800] #Tamaños del grid (número de dimensiones del problema de optimización)

if n_variables[0]%2 ==0:
    varsNLSE=n_variables[0]+1
else:
    varsNLSE=n_variables[0]


search_space = [
    Categorical([50, 500, 1000, 1500, 2000, 2500, 3000], name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
    Real(0.1, 0.8, name="alpha", prior='uniform'),                # Continuous values
    Real(0.1, 0.8, name="elite_factor", prior='uniform'),        # Continuous values
]

# Simulated function that runs the optimization and computes RMSE
def evaluate_algorithm(size_gen, alpha, elite_factor):
    """
    This function should run the actual optimization algorithm using the given parameters.
    It should return the mean MSE (MSE across the four first states of optimization problem).
    """
    ######################################################################################
    ######################################################################################
    #################### N L S E Non linear Schroedinger Equation ########################
    ######################################################################################
    print('\nNLSE IN PROGRESS\n')

    Resultados = np.empty((1, 9, 5)) 
    # 5 métricas [MSE, E, E%, GP_1000, t_exc]
    experimento = int(0)   
    L=2
    #El grid se define en para los tres problemas de optimización con el mismo número de variables
    if v%2 ==0:
        varsNLSE=v+1
    else:
        varsNLSE=v

    X_grid = np.linspace(start=-1, stop=1, num=varsNLSE).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=0 que son 1
    #X_train = X_grid[n_variables//2]
    X_train = np.array((0)).reshape(-1,1)
    Y_train = np.linspace(start=1, stop=1, num=1).reshape(-1,1) #Para que sea 2D array
    
    init_data = None #initialize_population(size_gen=3000, grid=X_grid, init_kernels=10, init_constraints=(X_train, Y_train))
    params = {'L':L}
    qegna = UMDAc(size_gen=size_gen, max_iter=300, dead_iter=60, n_variables= varsNLSE, lower_bound=0, upper_bound = 2, alpha=alpha, elite_factor=elite_factor, parallelize=True, init_data=init_data)#, lower_factor=0.01)
    #qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= varsNLSE, lower_bound=0, upper_bound = 2, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites], init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= varsNLSE, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= varsNLSE, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egparesultNLSE = qegna.minimize(NLSE, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 8, 4] = end-start
    MSE_NLSE = np.sum((1/np.cosh(X_grid[:,0]) - egparesultNLSE.sel_inds_hist[-1, 0, :-1])**2) / len(X_grid[:,0])
    MSE_NLSE = np.float64(MSE_NLSE)
    Resultados[experimento, 8, 0] = MSE_NLSE
    #setting non used columns to None
    Resultados[experimento, 8, 1:3] = None
    Resultados[experimento, :, 3] = None
    print(f'MSE: {MSE_NLSE}\n')    
    print('\nNLSE DONE\n')
    return MSE_NLSE


global call
call = 0

# Wrapper function for Bayesian Optimization
@use_named_args(search_space)
def objective(**params):
    global call
    call +=1
    print(f'\n##############################################\nBO call number {call} of 50\n##############################################\n\n')
    return evaluate_algorithm(**params)

for v in n_variables:
    # Run Bayesian Optimization
    print(f'\n##############################################\nBO for {v} variables\n##############################################\n\n')
    result = gp_minimize(
        func=objective,      # Objective function
        dimensions=search_space,  # Search space
        n_calls=50,         # Number of function evaluations
        random_state=42,     # For reproducibility,
        n_initial_points=12
    )

    # Best parameters found
    best_params = dict(zip(['size_gen', 'alpha', 'elite_factor', 'whites'], result.x))
    print("Best Parameters:", best_params)
    print("Best MSE (Min-Max Approach):", result.fun)


    
    ruta = f'UMDABO_params/{v}NLSE'
    with open(f'{ruta}.pkl', "wb") as file:
        pickle.dump(best_params, file)


    for keys in best_params.keys():
        print(f'Problema {keys}: {best_params[keys]}\n')



##############################################
BO for 50 variables
##############################################



##############################################
BO call number 1 of 50
##############################################



NLSE IN PROGRESS

IT:  0 	Best cost:  1041153.9496712278794
	NFEVALS = 300000 F = -4.5642510079814344173e-3198
	X = [[1.64633256 1.00008794 0.13103923 0.99910832 1.07794161 0.70892645
  1.31865371 1.58020519 1.32362862 1.23108602 1.04689826 0.63200364
  1.05467993 0.92628887 0.98529409 1.39299724 1.40744232 1.40966597
  0.68415472 0.47195908 0.4335866  0.90771766 0.93684724 1.16225682
  0.77970951 1.25000797 1.2245933  0.46752768 0.90404771 1.3386106
  1.71382163 1.22988613 0.77597718 0.97142852 1.03805401 1.14222566
  0.77526987 0.69047011 1.27287469 1.46100773 0.36359029 0.32616336
  0.61168068 1.30277621 1.29201313 1.28041061 0.84776441 0.89644101
  0.7750206  0.34995959 0.11922791]]
MSE: 0.15804502873024878


NLSE DONE


###########################

/home/dani/anaconda3/envs/EDAspy/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(1000), 0.8, 0.8] before, using random point [1000, 0.6269860040135552, 0.2780422554187313]
  warnings.warn(


IT:  0 	Best cost:  1308005.342991400306
	NFEVALS = 100000 F = -6.3521212328420085577e-3008
	X = [[1.22133632 1.19187501 1.34234863 0.91079822 0.70785104 1.57222554
  1.33713138 0.93325754 0.92065499 0.83314893 1.16315013 1.38824596
  0.82938125 0.46412579 1.55828771 1.42581545 1.41354603 0.79742656
  0.58726818 1.12050647 1.14283102 1.0594592  1.03607159 0.89971262
  0.94232374 0.6161437  1.23410969 0.8927214  1.12534478 1.07941098
  1.19427993 1.33853979 1.0821034  1.34139352 1.06547126 1.18107035
  1.16325095 1.28558291 0.59043743 0.45520193 0.92226026 1.55489417
  1.09350728 0.62851039 0.2402996  0.47090379 1.06722828 1.29314225
  1.39764048 1.10830063 0.51315351]]
MSE: 0.13952451273563166


NLSE DONE


##############################################
BO call number 24 of 50
##############################################



NLSE IN PROGRESS



/home/dani/anaconda3/envs/EDAspy/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(2500), 0.8, 0.8] before, using random point [3000, 0.3835891216927201, 0.4341029362898232]
  warnings.warn(


IT:  0 	Best cost:  1244795.4720116335393
	NFEVALS = 411000 F = -5.1939340213754661215e-3248
	X = [[1.00254165 0.97698012 1.55580831 1.50362576 1.38795822 1.25675629
  1.54482435 0.68601595 0.82074441 1.05528758 0.94026773 1.08756662
  0.71979658 0.79738437 0.92154467 1.03557044 1.25741176 2.06356848
  1.25917146 0.94991927 1.43710945 1.12954996 1.03447821 1.14935637
  1.18477184 1.06200524 1.32226042 1.30259244 0.80268493 0.81576668
  1.01974259 1.43521527 1.15511959 0.43397275 0.4585883  0.6521684
  1.66373618 1.43584873 1.06457008 1.5935228  0.92980675 0.4592928
  0.97144346 1.07185659 0.81977763 1.16739003 1.4013169  1.41230626
  0.96195665 0.64295251 0.7796001 ]]
MSE: 0.17235656767488888


NLSE DONE


##############################################
BO call number 25 of 50
##############################################



NLSE IN PROGRESS



/home/dani/anaconda3/envs/EDAspy/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(2500), 0.8, 0.8] before, using random point [1000, 0.25599478448243335, 0.3300620629641787]
  warnings.warn(


... (13 outputs left out)

/home/dani/anaconda3/envs/EDAspy/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(1000), 0.8, 0.8] before, using random point [2500, 0.22113420630704633, 0.690393953570877]
  warnings.warn(


IT:  0 	Best cost:  6619337474.0574725997
	NFEVALS = 200000 F = -0.0
	X = [[ 8.30958013e-01  1.16839894e+00  2.08433978e+00  1.43840108e+00
   1.73816717e+00 -2.51363645e-01  8.23978379e-01  2.19615265e+00
   1.33069569e+00 -2.24670530e-02  6.93167719e-01  1.06916281e+00
   1.17302530e+00  5.18686010e-01  1.37281397e-01  2.70187247e-01
   2.17196007e+00  9.50115099e-01  1.00039231e+00  1.30508219e+00
   1.56750028e+00  2.74273756e+00  1.21387862e+00  4.31240960e-01
   7.38737924e-01  6.19211145e-01  1.09241868e-01  2.07080588e+00
   1.72248277e+00  1.41332202e+00  1.29036873e-01  6.44594720e-01
   6.20931660e-01  1.24750804e+00  1.19632752e+00  2.42004933e+00
   1.98977478e-01  1.06510296e+00  1.49093394e+00  5.91292446e-01
   1.30758834e+00  3.45169915e-01  1.00072935e+00  2.57426679e-01
   7.04405778e-01  1.42317446e+00  1.20523231e+00  7.64765590e-01
  -2.73167509e-01  1.29229172e+00  6.87574231e-01  1.16836650e+00
   1.43670500e+00  2.56445560e-02  1.52431829e+00  7.14854919e-01
  

/home/dani/anaconda3/envs/EDAspy/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(1000), 0.8, 0.8] before, using random point [1000, 0.4146827945926731, 0.31580127813134495]
  warnings.warn(


IT:  0 	Best cost:  7008473746.4347171336
	NFEVALS = 80000 F = -0.0
	X = [[ 1.71507016e+00 -3.89139174e-01  9.38469265e-01  1.94248495e+00
   7.40090559e-01  7.46039837e-01  1.28323067e+00  1.54671101e+00
   2.52431911e-01  6.35890766e-01  1.09574862e+00  2.11614310e+00
   1.45614751e+00  1.05520242e+00  1.32749473e+00  1.62858442e-01
   4.54904190e-01  5.07554349e-01  1.54738196e+00  1.14498977e+00
   1.05751822e+00  3.93730712e-01  1.48832898e+00 -8.85677936e-02
   7.38116618e-01  9.18508840e-01  7.06545060e-01  8.61260421e-01
   1.08848719e+00  6.98596933e-02  8.58332744e-01  8.68154218e-01
   1.21775571e+00  5.29229932e-01  7.13118445e-01  6.48416046e-01
   1.33283200e-01  1.99075789e+00  6.57897331e-01  6.10866508e-01
   7.40018888e-01  1.92921099e+00  1.30639242e+00  1.52781487e+00
   6.64982716e-03  1.04357399e+00  1.54231647e+00  1.42411202e+00
   1.11273581e+00  1.48051146e+00  1.33676950e+00  1.41635787e+00
  -1.90054074e-02  7.34029975e-01  8.00045448e-01  7.00557182e-01
   

## **Infinite Well**

In [2]:
def _normalize(dataset:np.array, grid_size: float):
        '''
        Normalizes the solutions i.e. multiply by a constant so that the integral of the solution^2 = 1.
        Be aware that integration depends on problem definition and size!!!!
        '''
        #grid_size = L/variables     dx of integral
        #grid_size = 5/(self.n_variables-1)

        #Assuming a constant size grid we know grid size independently of problem parameters (better code-wise)
        #self.grid_size = grid_X[-1,0] - grid_X[-2, 0]

        for ind in range(len(dataset)):
                integral = 0
                for var in range(dataset.shape[1]):
                    integral = integral + (dataset[ind,:][var]**2)*grid_size ##Tener en cuenta posibles cambios en tamaño grid!!! L caja = 5 

                A = (1/integral)**0.5
                dataset[ind, :] = A * dataset[ind, :]


In [3]:
import numpy as np
import sys
import os
# Get the absolute path of the main directory
main_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# Add the main directory to sys.path
sys.path.append(main_dir)
from OptFunctions.oneDwellfunctions import *
import time
#Para Bayesian optimization de parámetros
from skopt import gp_minimize
from skopt.space import Integer, Real, Categorical
from skopt.utils import use_named_args
#Para usar EGNA
from EDAspy.optimization import EGNA
from EDAspy.optimization import UMDAc
from EDAspy.optimization import EMNA
#Para procesos Gaussianos
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
import pickle

n_variables = [50, 425, 800] #Tamaños del grid (número de dimensiones del problema de optimización)



search_space = [
    Categorical([50, 500, 1000, 1500, 2000, 2500, 3000], name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
    Real(0.1, 0.8, name="alpha", prior='uniform'),                # Continuous values
    Real(0.1, 0.8, name="elite_factor", prior='uniform'),        # Continuous values
]

# Simulated function that runs the optimization and computes RMSE
def evaluate_algorithm(size_gen, alpha, elite_factor):
    """
    This function should run the actual optimization algorithm using the given parameters.
    It should return the mean MSE (MSE across the four first states of optimization problem).
    """
    Resultados = np.empty((1, 9, 5)) 
    # 5 métricas [MSE, E, E%, GP_1000, t_exc]
    experimento = int(0)    

    ######################################################################################
    ######################################################################################
    ############## POZO DE POTENCIAL INFINITO 1D WELL ####################################
    ######################################################################################

    print('\nInfinite Well IN PROGRESS\n')
    L=5
    #Energías teóricas
    Eg = (2*np.pi)**2/(8*L**2*1)
    En=[]
    for i in range(4):
        e=(i+1)**2*Eg
        En.append(e)

    ## Problem grid definition
    X_grid = np.linspace(start=0, stop=L, num=v).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=0 x=5 que son 0
    X_train = X_grid[[0,-1]]
    Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array
    init_data = None #initialize_population(size_gen=3000, grid=X_grid, init_kernels=10, init_constraints=(X_train, Y_train))
    grid_size = X_grid[-1,0] - X_grid[-2, 0]
    params = {'L':L}

    #Soluciones teóricas
    ### Soluciones analíticas
    # def Psi_n(n):
    #     return((2/L)**0.5*np.sin(n*np.pi*np.linspace(0, L, n_variables)/L))

    APsisWell = {}
    for n in range(4):
        APsisWell[f'APsi{n}'] = (2/L)**0.5*np.sin((n+1)*np.pi*np.linspace(0, L, v)/L)
    ###############################    
    #Ground state n=1
    ###############################
    print(f'F should be approx -0.9618')
    qegna = UMDAc(size_gen=size_gen, max_iter=300, dead_iter=60, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, init_data=init_data)
    #qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites], init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultgroundWell = qegna.minimize(oneD_wellExpE, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 4, 4] = end-start
    _normalize(dataset=egnaresultgroundWell.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    Psi_fund=egnaresultgroundWell.sel_inds_hist[-1, 0, :-1]
    E_fund = np.sqrt(-np.log(-egnaresultgroundWell.sel_inds_hist[-1,0,-1])) # def exponencial de función de coste la energía se determina a partir del fitness así.
    #En este caso no hay constante multiplicando exponente exponencial
    Resultados[experimento, 4, 1] = E_fund
    Resultados[experimento, 4, 2] = abs(E_fund-En[0])/En[0]*100 

    params = {'L':L, 'E': E_fund, 'Psi': Psi_fund}
    print(f'\n  |Psi0> done\n')

    ###############################   
    #1excitated state n=2
    ###############################   
    print(f'F should be approx -0.7042')
    qegna = UMDAc(size_gen=size_gen, max_iter=300, dead_iter=60, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, init_data=init_data)
    #qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites], init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultWellexc1 = qegna.minimize(oneD_wellExpEexcitated, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 5, 4] = end-start
    _normalize(dataset=egnaresultWellexc1.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    Psi_1=egnaresultWellexc1.sel_inds_hist[-1, 0, :-1]
    E1 = np.sqrt(-np.log(-egnaresultWellexc1.sel_inds_hist[-1,0,-1]))
    Resultados[experimento, 5, 1] = E_fund + E1
    Resultados[experimento, 5, 2] = abs(E_fund + E1 -En[1])/En[1]*100

    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1}
    print(f'\n  |Psi1> done\n')

    ###############################   
    #2nd excitated state n=3
    ###############################   
    print(f'F should be approx -0.3775')
    qegna = UMDAc(size_gen=size_gen, max_iter=300, dead_iter=60, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, init_data=init_data)
    #qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites], init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultWellexc2 = qegna.minimize(oneD_wellExpEexcitated3, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 6, 4] = end - start
    _normalize(dataset=egnaresultWellexc2.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    Psi_2=egnaresultWellexc2.sel_inds_hist[-1, 0, :-1]
    E2 = np.sqrt(-np.log(-egnaresultWellexc2.sel_inds_hist[-1,0,-1]))
    Resultados[experimento, 6, 1] = E_fund + E1 + E2
    Resultados[experimento, 6, 2] = abs(E_fund + E1 +E2 -En[2])/En[2]*100

    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2}
    print(f'\n  |Psi2> done\n')

    ###############################   
    #3rd excitated state n=4
    ###############################   
    print(f'F should be approx -0.1482')
    qegna = UMDAc(size_gen=size_gen, max_iter=300, dead_iter=60, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, init_data=init_data)
    #qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites], init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.6, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultWellexc3 = qegna.minimize(oneD_wellExpEexcitated4, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 7, 4] = end-start
    _normalize(dataset=egnaresultWellexc3.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    Psi_3 = egnaresultWellexc3.sel_inds_hist[-1, 0, :-1]
    E3 = np.sqrt(-np.log(-egnaresultWellexc3.sel_inds_hist[-1,0,-1]))
    Resultados[experimento, 7, 1] = E_fund + E1 + E2 + E3
    Resultados[experimento, 7, 2] = abs(E_fund + E1 +E2+E3 -En[3])/En[3]*100

    params = {'L':L, 'E0': E_fund, 'Psi0': Psi_fund, 'E1':E1, 'Psi1': Psi_1, 'E2': E2, 'Psi2': Psi_2, 'E3':E3, 'Psi3': Psi_3}
    print(f'\n  |Psi3> done\n')

    #Cálculos de métricas error MSE de cada estado con estado teórico
    MSEs = []
    for i in range(4):
        mse = min(np.mean((params[f'Psi{i}'] - APsisWell[f'APsi{i}'])**2), np.mean((-1*params[f'Psi{i}'] - APsisWell[f'APsi{i}'])**2))
        Resultados[experimento, i+4, 0] = mse
        MSEs.append(mse)

    MSEmean = np.mean(MSEs)
    print('\nInfinite Well DONE\n')
    return MSEmean


global call
call = 0

# Wrapper function for Bayesian Optimization
@use_named_args(search_space)
def objective(**params):
    global call
    call +=1
    print(f'\n##############################################\nBO call number {call} of 150\n##############################################\n\n')
    return evaluate_algorithm(**params)

for v in n_variables:
    # Run Bayesian Optimization
    print(f'\n##############################################\nBO for {v} variables\n##############################################\n\n')
    result = gp_minimize(
        func=objective,      # Objective function
        dimensions=search_space,  # Search space
        n_calls=50,         # Number of function evaluations
        random_state=42,     # For reproducibility,
        n_initial_points=12
    )

    # Best parameters found
    best_params = dict(zip(['size_gen', 'alpha', 'elite_factor', 'whites'], result.x))
    print("Best Parameters:", best_params)
    print("Best MSE (Min-Max Approach):", result.fun)


    
    ruta = f'UMDABO_params/{v}Well'
    with open(f'{ruta}.pkl', "wb") as file:
        pickle.dump(best_params, file)


    for keys in best_params.keys():
        print(f'Problema {keys}: {best_params[keys]}\n')



##############################################
BO for 50 variables
##############################################



##############################################
BO call number 1 of 150
##############################################



Infinite Well IN PROGRESS

F should be approx -0.9618
IT:  0 	Best cost:  2022.2592731690352
	NFEVALS = 282500 F = -1.9689162461942896e-05
	X = [[-1.41818376  1.08006379 -0.43188194 -0.34412596 -0.23301775 -0.45131315
  -0.32346254 -0.41532105 -0.05530327 -0.01228668 -0.28300836 -0.31841792
  -0.00366393  0.28452885  0.63249945 -0.2861149   0.03297651 -0.46841267
  -0.27616153  0.33099067  0.35811939  0.0794765   0.30263499  0.32736315
   0.10219753  0.55360793  0.29456325  0.91279357  0.45278798  0.50261256
  -0.17785845 -0.25174625  0.09103534  0.14352306  0.08924608 -0.15734332
   0.29208988  0.28979654 -0.23559034 -0.0322194  -0.32224977  0.39327569
  -0.04210507  0.46372878  0.11506562  0.27779147  0.11911657 -0.43139728
   0.08086967 -2.06539904

/tmp/ipykernel_1365641/410315612.py:89: RuntimeWarning: divide by zero encountered in log
  E_fund = np.sqrt(-np.log(-egnaresultgroundWell.sel_inds_hist[-1,0,-1])) # def exponencial de función de coste la energía se determina a partir del fitness así.


IT:  0 	Best cost:  inf
	NFEVALS = 200000 F = -0.0
	X = [[ 1.60128743e-01 -8.26663636e-01  1.48460481e-01 -1.14235381e-01
  -1.40826240e+00 -5.19483046e-01  1.33499211e-01  6.89424042e-01
   1.27240918e-01  4.60060138e-01  2.39482247e-01 -5.82531262e-02
  -1.85168290e-01 -4.87759905e-02 -4.90430469e-01 -1.06194538e-01
  -1.42849428e-01  5.78248666e-01 -1.45769571e+00 -1.19642432e-01
   6.29597864e-01  2.31501350e-01  5.83187355e-02 -1.08589789e+00
  -8.11093491e-01 -4.01506935e-01 -1.05380199e+00  1.36640928e-01
   7.78050441e-01  4.04260216e-01  1.97728314e-01 -9.69158748e-02
   1.84380687e-01 -2.51697565e-01  4.87141436e-01  4.15119184e-01
   1.84882828e-01 -1.05192298e+00 -4.25838416e-02  7.71687708e-01
   4.00268170e-01 -1.19929210e-01 -2.85626168e-01 -1.04656604e+00
  -3.53534230e-01 -6.05435629e-01 -1.03207716e+00 -9.35812789e-01
  -3.76802775e-01  3.66548306e-01 -7.90912153e-02  4.13958863e-01
   5.09414637e-01 -9.99625272e-02  7.36319744e-01 -1.87655613e-01
  -7.67840453e-01  8

/tmp/ipykernel_1365641/410315612.py:111: RuntimeWarning: divide by zero encountered in log
  E1 = np.sqrt(-np.log(-egnaresultWellexc1.sel_inds_hist[-1,0,-1]))


IT:  0 	Best cost:  inf
	NFEVALS = 200000 F = -0.0
	X = [[-2.70003196e-01  1.74194542e-01  1.39345318e-01  4.37699852e-01
   2.29738646e-01 -2.82188498e-01  2.67275404e-01 -3.13260923e-01
   4.47194355e-01 -6.19779122e-03 -3.83730719e-01  4.09626896e-01
  -1.00027393e-01 -1.30416000e-01 -4.45597947e-01  1.85571443e-01
   1.43056195e-03  1.11279046e+00  2.30433450e-01 -4.57299655e-01
  -7.52869554e-02 -1.96420125e-01  1.15986244e+00  5.36649853e-01
   1.32156552e+00  2.99024746e-01 -4.93377557e-01  5.37973117e-01
  -1.43004380e-01 -1.40112701e+00  1.45691295e-01 -1.03356368e-01
  -5.18790247e-01 -3.89517062e-01  4.76370104e-01 -5.86343549e-01
  -4.26827276e-01  7.19618476e-01 -6.53109738e-01 -2.77456243e-01
  -1.33564375e+00  4.70381466e-01  6.84442893e-01  3.17195542e-01
   2.66306345e-01  2.14371125e-01 -1.75559175e-01 -4.00500349e-01
   2.87057504e-01  2.63979076e-01 -4.88793231e-01 -1.45625762e-01
  -2.60823118e-01 -1.10891585e-01 -1.02093554e-01 -9.74626568e-01
   3.85951181e-01  2

/tmp/ipykernel_1365641/410315612.py:132: RuntimeWarning: divide by zero encountered in log
  E2 = np.sqrt(-np.log(-egnaresultWellexc2.sel_inds_hist[-1,0,-1]))


... (615 outputs left out)

/tmp/ipykernel_1365641/410315612.py:132: RuntimeWarning: divide by zero encountered in log
  E2 = np.sqrt(-np.log(-egnaresultWellexc2.sel_inds_hist[-1,0,-1]))


IT:  0 	Best cost:  inf
	NFEVALS = 160000 F = -0.0
	X = [[ 4.75077148e-01  5.07407868e-01  7.67892877e-01  5.56800414e-01
   6.27127264e-01 -5.31768034e-01  4.38207156e-01 -1.22915492e+00
   1.49199741e-02 -4.00297942e-01  3.22237687e-01 -4.64120339e-01
  -2.35916405e-01  1.00115469e+00  2.17221176e-01 -5.47030316e-01
   5.68751887e-01 -4.85948469e-01 -1.96008289e-02 -6.72941364e-01
   4.76835814e-01  4.18031600e-01  2.71766515e-03  2.64846761e-02
   6.68604550e-01  1.46267305e-01  3.63489357e-01 -1.87453559e-01
   5.74410220e-01 -5.95746735e-01  1.58090122e+00  4.84849571e-01
   4.60002715e-01  1.46345948e-01  3.69286382e-01  3.81506708e-01
   1.16539680e+00  3.10682477e-02  4.50590743e-01 -3.95919313e-01
  -8.06426273e-01  4.54255320e-02 -1.80539924e-01 -4.83368868e-02
   1.42391331e+00  3.23578711e-03  5.50826152e-01  5.27904951e-01
   1.21830247e-01  1.30315792e+00 -6.89903361e-01 -3.97613629e-01
  -3.03053754e-01 -9.86965256e-01 -1.67152946e-01  6.73874992e-01
  -2.51673585e-01 -1

/tmp/ipykernel_1365641/410315612.py:153: RuntimeWarning: divide by zero encountered in log
  E3 = np.sqrt(-np.log(-egnaresultWellexc3.sel_inds_hist[-1,0,-1]))



  |Psi3> done


Infinite Well DONE

Best Parameters: {'size_gen': np.int64(1000), 'alpha': 0.7654188173804739, 'elite_factor': 0.11368547610918285}
Best MSE (Min-Max Approach): 0.3750581829464372
Problema size_gen: 1000

Problema alpha: 0.7654188173804739

Problema elite_factor: 0.11368547610918285



## **Quantum Oscillator**

In [4]:
def _normalize(dataset:np.array, grid_size: float):
        '''
        Normalizes the solutions i.e. multiply by a constant so that the integral of the solution^2 = 1.
        Be aware that integration depends on problem definition and size!!!!
        '''
        #grid_size = L/variables     dx of integral
        #grid_size = 5/(self.n_variables-1)

        #Assuming a constant size grid we know grid size independently of problem parameters (better code-wise)
        #self.grid_size = grid_X[-1,0] - grid_X[-2, 0]

        for ind in range(len(dataset)):
                integral = 0
                for var in range(dataset.shape[1]):
                    integral = integral + (dataset[ind,:][var]**2)*grid_size ##Tener en cuenta posibles cambios en tamaño grid!!! L caja = 5 

                A = (1/integral)**0.5
                dataset[ind, :] = A * dataset[ind, :]


In [5]:
import numpy as np
import sys
import os
# Get the absolute path of the main directory
main_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
# Add the main directory to sys.path
sys.path.append(main_dir)
from OptFunctions.oneDQHarmOscillatorfunctions import *
import time
#Para Bayesian optimization de parámetros
from skopt import gp_minimize
from skopt.space import Integer, Real, Categorical
from skopt.utils import use_named_args
#Para usar EGNA
from EDAspy.optimization import EGNA
from EDAspy.optimization import UMDAc
from EDAspy.optimization import EMNA
#Para procesos Gaussianos
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
import pickle

n_variables = [50, 425, 800] #Tamaños del grid (número de dimensiones del problema de optimización)


search_space = [
    Categorical([50, 500, 1000, 1500, 2000, 2500, 3000], name="size_gen"),  # Discrete values Integer(50, 1000, name="size_gen") will take integer values in the interval 50, 1000
    Real(0.1, 0.8, name="alpha", prior='uniform'),                # Continuous values
    Real(0.1, 0.8, name="elite_factor", prior='uniform'),        # Continuous values
]

# Simulated function that runs the optimization and computes RMSE
def evaluate_algorithm(size_gen, alpha, elite_factor):
    """
    This function should run the actual optimization algorithm using the given parameters.
    It should return the mean MSE (MSE across the four first states of optimization problem).
    """
    Resultados = np.empty((1, 9, 5)) 
    # 5 métricas [MSE, E, E%, GP_1000, t_exc]
    experimento = int(0) 
    ##########################################################################################
    ##########################################################################################
    ########################### OSCILADOR ARMÓNICO ###########################################
    ##########################################################################################
    print('Quantum Harmonic Oscillator IN PROGRESS\n')
    #Problem grid definition
    X_grid = np.linspace(start=-4, stop=4, num=v).reshape(-1,1) #para que sean n_variables filas y 1 columna
    #Ahora definimos las observaciones que hemos hecho de la función. En este caso son las condiciones de contorno del problema
    #Los datos que conocemos son las amplitudes en x=-4 (-inf) x=4 (inf) que son 0
    X_train = X_grid[[0,-1]]
    Y_train = np.linspace(start=0, stop=0, num=2).reshape(-1,1) #Para que sea 2D array

    #Analytical solutions definitions
    #Ground state
    APsi0 = np.exp(-X_grid**2/2)
    integral =0
    for i in range(len(APsi0)):
        integral = integral + (APsi0[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi0)-1))

    B = (1/integral)**0.5
    APsi0 = B * APsi0

    #1st excitated state
    APsi1 = np.exp(-X_grid**2/2)*2*X_grid
    integral =0
    for i in range(len(APsi1)):
        integral = integral + (APsi1[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi1)-1))

    B = (1/integral)**0.5
    APsi1 = -1 * B * APsi1 #signo menos para correspondencia con solución algoritmo (simetrización)

    #2nd excitated state
    APsi2 = np.exp(-X_grid**2/2)*(4*X_grid**2-2)
    integral =0
    for i in range(len(APsi2)):
        integral = integral + (APsi2[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi2)-1))

    B = (1/integral)**0.5
    APsi2 = B * APsi2

    #3rd excitated state
    APsi3 = np.exp(-X_grid**2/2)*(8*X_grid**3-12*X_grid)
    integral =0
    for i in range(len(APsi3)):
        integral = integral + (APsi3[i]**2)*((X_grid[-1]-X_grid[0])/(len(APsi3)-1))

    B = (1/integral)**0.5
    APsi3 = -1 * B * APsi3 #signo menos para correspondencia con solución algoritmo (simetrización)

    APsis = {}
    APsis['APsi0'] = APsi0[:, 0]
    APsis['APsi1'] = APsi1[:, 0]
    APsis['APsi2'] = APsi2[:, 0]
    APsis['APsi3'] = APsi3[:, 0]
    X_grid = X_grid[:, 0]
    grid_size = X_grid[-1] - X_grid[-2]

    ##########################################################################################################
    ##########################################################################################################

    ### Ahora el código con EGPA que calcula los 4 primeros estados excitados del problema oscilador armónico
    params = {'x':X_grid}
    #Ground state
    print(f'F should be approx -0.082')
    qegna = UMDAc(size_gen=size_gen, max_iter=300, dead_iter=60, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True)
    #qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites])#, init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultQHOground = qegna.minimize(oneDHOExpE, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 0, 4] = end-start
    _normalize(dataset=egnaresultQHOground.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    #Se guardan resultados de algoritmo para siguiente estado excitado
    Psi_ground=egnaresultQHOground.sel_inds_hist[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E0 = np.sqrt(-np.log(-egnaresultQHOground.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así 
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground}
    Resultados[experimento, 0, 1] = E0
    Resultados[experimento, 0, 2] = abs(E0-0.5)/0.5*100

    print(f'\n  |Psi0> done\n')

    #1er estado excitado
    print(f'F should be approx -4.54e-5')
    qegna = UMDAc(size_gen=size_gen, max_iter=300, dead_iter=60, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True)
    #qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites])#, init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultQHO1 = qegna.minimize(oneDHOExpE1, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 1, 4] = end-start
    _normalize(dataset=egnaresultQHO1.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    Psi1=egnaresultQHO1.sel_inds_hist[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E1 = np.sqrt(-np.log(-egnaresultQHO1.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    Resultados[experimento, 1, 1] = E0 + E1
    Resultados[experimento, 1, 2] = abs(E0 + E1 -1.5)/1.5*100
    
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1}
    print(f'\n  |Psi1> done\n')

    #2o estado excitado
    print(f'F should be approx -4.54e-5')
    qegna = UMDAc(size_gen=size_gen, max_iter=300, dead_iter=60, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True)
    #qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites])#, init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultQHO2 = qegna.minimize(oneDHOExpE2, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 2, 4] = end-start
    _normalize(dataset=egnaresultQHO2.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    Psi2=egnaresultQHO2.sel_inds_hist[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E2 = np.sqrt(-np.log(-egnaresultQHO2.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    Resultados[experimento, 2, 1] = E0 + E1 + E2
    Resultados[experimento, 2, 2] = abs(E0 + E1 + E2 -2.5)/2.5*100
    
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2 }
    print(f'\n  |Psi2> done\n')

    #3er estado excitado
    print(f'F should be approx -4.54e-5')
    qegna = UMDAc(size_gen=size_gen, max_iter=300, dead_iter=60, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True)
    #qegna = EGNA(size_gen=size_gen, max_iter=150, dead_iter=50, n_variables= v, lower_bound=-1, upper_bound = 1, alpha=alpha, elite_factor=elite_factor, parallelize=True, white_list=ARCOS[whites])#, init_data=init_data)
    #qegna = EMNA(size_gen=3200, max_iter=600, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=0.4, parallelize=True)#, lower_factor = 0.3)#, upper_factor=0.6)#, lower_factor=0.01)
    #qegna = SPEDA(size_gen=3200, max_iter=800, dead_iter=201, n_variables= v, lower_bound=-0.5, upper_bound = 0.5, alpha=alpha, parallelize=True, l=0, black_list=blacks) 
    start = time.time()
    egnaresultQHO3 = qegna.minimize(oneDHOExpE3, params, exp_from = 20, output_runtime=False)
    end = time.time()
    Resultados[experimento, 3, 4] = end-start
    _normalize(dataset=egnaresultQHO3.sel_inds_hist[-1, :,:-1], grid_size=grid_size)
    Psi3=egnaresultQHO3.sel_inds_hist[-1, 0, :-1]
    cons = 10 ### FACTOR CONSTANTE que MULTIPLICA EL EXPONENTE EN FUNCIÓN DE COSTE 
    E3 = np.sqrt(-np.log(-egnaresultQHO3.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así
    Resultados[experimento, 3, 1] = E0 + E1 + E2 + E3
    Resultados[experimento, 3, 2] = abs(E0 + E1 + E2 + E3 -3.5)/3.5*100
    
    params = {'x':X_grid, 'E0': E0, 'Psi0': Psi_ground, 'E1':E1, 'Psi1': Psi1, 'E2':E2, 'Psi2': Psi2, 'E3': E3, 'Psi3': Psi3 }
    print(f'\n  |Psi3> done\n')

    #Cálculos de métricas error MSE de cada estado con estado teórico
    MSEs = []
    for i in range(4):
        mse = min(np.mean((params[f'Psi{i}'] - APsis[f'APsi{i}'])**2), np.mean((-1*params[f'Psi{i}'] - APsis[f'APsi{i}'])**2))
        Resultados[experimento, i, 0] = mse
        MSEs.append(mse)

    MSEmean = np.mean(MSEs)
    print('\nInfinite Oscillator DONE\n')
    return MSEmean


global call
call = 0

# Wrapper function for Bayesian Optimization
@use_named_args(search_space)
def objective(**params):
    global call
    call +=1
    print(f'\n##############################################\nBO call number {call} of 150\n##############################################\n\n')
    return evaluate_algorithm(**params)

for v in n_variables:
    # Run Bayesian Optimization
    print(f'\n##############################################\nBO for {v} variables\n##############################################\n\n')
    result = gp_minimize(
        func=objective,      # Objective function
        dimensions=search_space,  # Search space
        n_calls=50,         # Number of function evaluations
        random_state=42,     # For reproducibility,
        n_initial_points=12
    )

    # Best parameters found
    best_params = dict(zip(['size_gen', 'alpha', 'elite_factor', 'whites'], result.x))
    print("Best Parameters:", best_params)
    print("Best MSE (Min-Max Approach):", result.fun)


    
    ruta = f'UMDABO_params/{v}Osc'
    with open(f'{ruta}.pkl', "wb") as file:
        pickle.dump(best_params, file)


    for keys in best_params.keys():
        print(f'Problema {keys}: {best_params[keys]}\n')



##############################################
BO for 50 variables
##############################################



##############################################
BO call number 1 of 150
##############################################


Quantum Harmonic Oscillator IN PROGRESS

F should be approx -0.082
IT:  0 	Best cost:  476.64608127389175
	NFEVALS = 245000 F = -8.250551625743791e-106
	X = [[ 3.09734245 -0.31901747  0.44074606  0.47591144  0.12148385  0.05836221
  -0.03320718 -0.34846163 -0.29249987 -0.24118888 -0.37674368  0.2189894
   0.13815237 -0.49781038  0.06990579 -0.50092284  0.05353281  0.19407508
   0.81535737  0.06725967  0.34638152 -0.14761207 -0.20588121 -0.16924717
  -0.35177332  0.18588046 -0.17972608 -0.05264027  0.20371428 -0.22341741
   0.00776309  0.48327635  0.39357569  0.51439802  0.14161358 -0.14861023
   0.01330017 -0.43717152 -0.388736   -0.09875038 -0.56575202  0.04489235
  -0.01566698  0.19493035  0.20277673  0.29873907 -0.06076032  0.59629667
   0.4019501  

/tmp/ipykernel_1365641/1741729273.py:117: RuntimeWarning: divide by zero encountered in log
  E0 = np.sqrt(-np.log(-egnaresultQHOground.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así


IT:  0 	Best cost:  inf
	NFEVALS = 4000 F = -0.0
	X = [[ 0.50841314 -0.01740199  0.07128281  0.30998139 -0.39477717 -0.1351173
   0.59641467 -0.21154809 -0.022645    0.23273079 -0.33509777  1.54322412
  -0.18524879  0.27520461 -0.23554433 -0.20936573 -0.28846717 -0.55235452
  -0.36676286 -0.14057885 -0.81457818 -1.33882986  0.37336408 -0.40474403
  -0.89919088  1.18290187  0.54359282 -1.07019325  1.29796927 -0.81862861
   0.03728035  0.55775453 -1.18718622  0.42158316  0.28891703 -0.73880134
   0.59469301 -0.96356062  1.00131563  0.28946816 -0.13639233  0.40877474
   0.93944687 -0.803909   -0.60391473  0.52883621 -0.79955515 -0.38255588
  -1.05440827 -0.90916642]]

  |Psi1> done

F should be approx -4.54e-5


/tmp/ipykernel_1365641/1741729273.py:137: RuntimeWarning: divide by zero encountered in log
  E1 = np.sqrt(-np.log(-egnaresultQHO1.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así


IT:  0 	Best cost:  inf
	NFEVALS = 4000 F = -0.0
	X = [[-0.23270139  0.00503875  0.36853525 -1.76256697  0.42660419 -0.22851699
   0.82402067 -0.13461559  0.77677637  1.16787207  0.47052446  0.58037326
  -0.66792994 -1.58139603 -1.05250245 -1.02359385  0.40056257 -0.30773424
  -0.769994   -1.50624485  0.25242119  0.41302573 -1.18727609  0.47570763
   1.37797662  0.58005045  0.42676658 -0.16796138  1.23991057  0.61180162
   0.26017061  0.19518802  0.95366588  0.26602947  1.01232409  0.18718004
   0.47079548 -0.55005763  0.14652738  0.39816246  0.46129493  0.46990038
   0.41107816  0.20908811 -0.94878415  1.29581716 -0.33875109  0.28753781
  -0.16172575 -1.25957218]]

  |Psi2> done

F should be approx -4.54e-5


/tmp/ipykernel_1365641/1741729273.py:157: RuntimeWarning: divide by zero encountered in log
  E2 = np.sqrt(-np.log(-egnaresultQHO2.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así


... (623 outputs left out)

/tmp/ipykernel_1365641/1741729273.py:157: RuntimeWarning: divide by zero encountered in log
  E2 = np.sqrt(-np.log(-egnaresultQHO2.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así


IT:  0 	Best cost:  inf
	NFEVALS = 80000 F = -0.0
	X = [[-1.04897770e+00 -3.36796110e-01  7.14595355e-01 -1.78385177e-01
   4.12561863e-01  1.93327695e-01 -5.10637110e-01 -1.32194980e-01
   5.18117430e-01  2.71366147e-01  6.41945059e-01 -5.27570132e-01
  -1.26224668e-01 -1.15998721e-01 -1.11392953e+00  5.38443755e-01
  -9.13620253e-02 -6.87936660e-01  6.37863244e-02 -2.64925801e-02
   2.70172361e-02  6.76530330e-01  3.18740649e-01 -1.26107982e-01
   2.58242960e-01 -5.17879325e-01  9.06969011e-01  7.26187850e-01
   1.20858935e+00 -1.33427655e-01 -2.28960904e-02  7.02810974e-01
   4.66816264e-01  8.52504092e-01  2.27375828e-01  1.08710213e-01
   5.12168594e-01  9.73732160e-02  7.82332172e-01 -1.26236535e-01
   2.62767751e-01  1.86397874e-01 -6.86590655e-01 -4.97440414e-01
  -1.92362234e-01 -8.28910140e-01 -5.30330588e-01  1.70752343e-01
  -8.85156036e-01  7.28903367e-01  2.26199854e-01 -1.77832207e-01
   2.27453687e-02  6.49461310e-01  2.86010694e-01 -2.72399063e-01
   6.75453495e-01 -9.

/tmp/ipykernel_1365641/1741729273.py:177: RuntimeWarning: divide by zero encountered in log
  E3 = np.sqrt(-np.log(-egnaresultQHO3.sel_inds_hist[-1,0,-1])/cons) # con definición exponencial de función de coste la energía se determina a partir del fitness así



  |Psi3> done


Infinite Oscillator DONE

Best Parameters: {'size_gen': 50, 'alpha': 0.4673422621808725, 'elite_factor': 0.37990268020067897}
Best MSE (Min-Max Approach): 0.23481700706835168
Problema size_gen: 50

Problema alpha: 0.4673422621808725

Problema elite_factor: 0.37990268020067897

